# S5a — ANSUR II formulas and the chest/waist mix test (Colab, CPU is enough)

specs/s5a-ansur/spec.md (approved 25 Nov 2026). Trains no network: fits straight-line formulas on ANSUR II
and tests, on BodyM validation people, whether mixing ANSUR into chest/bust and waist beats r4 alone.

**Before you start:** download the two ANSUR II files from https://www.openlab.psu.edu/ansur2/ (Downloads →
ANSUR II Data): `ANSUR II MALE Public.csv` and `ANSUR II FEMALE Public.csv`. Put both in
`MyDrive/anthro/ansur2/` on Google Drive. They never go into the repository.


## 1. Drive and data (same as notebook 04)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE = "/content/drive/MyDrive/anthro"
DRIVE_ARCHIVE = "/content/drive/MyDrive/datasets/bodym.tar"
DATA_DIR = "/content/bodym"
os.makedirs(DRIVE, exist_ok=True)

if os.path.exists(DATA_DIR):
    print("Data already on this runtime.")
elif os.path.exists(DRIVE_ARCHIVE):
    !tar -xf "{DRIVE_ARCHIVE}" -C /content
else:
    !pip -q install awscli
    !aws s3 sync --no-sign-request s3://amazon-bodym/ {DATA_DIR} --only-show-errors
    os.makedirs(os.path.dirname(DRIVE_ARCHIVE), exist_ok=True)
    !tar -cf "{DRIVE_ARCHIVE}" -C /content bodym

## 2. Get the code

In [ ]:
BRANCH = "s5a-ansur"   # set to "main" once merged
REPO = "https://github.com/Anthropometry/Anthropometry_Accademic_Model.git"
!rm -rf /content/repo && git clone -q --branch {BRANCH} {REPO} /content/repo
!pip -q install -e /content/repo
!git -C /content/repo log -1 --format="commit %h  %s"
GEO, CACHE = f"{DRIVE}/geo", f"{DRIVE}/cache"
RUN = f"{DRIVE}/runs/r4"
ANSUR = f"{DRIVE}/ansur2"
FORMULAS = f"{ANSUR}/ansur_formulas.json"

# A package installed in a running Colab session is only importable after a restart;
# pointing Python at the code directly avoids that.
import sys
sys.path.insert(0, "/content/repo/src")


## 3. Check the ANSUR files are on Drive

In [ ]:
from pathlib import Path
from anthro import ansur
for sex in ("male", "female"):
    try:
        print("found  ", ansur.find_file(Path(ANSUR), sex).name)
    except ansur.AnsurError as e:
        print("MISSING", e)


## 4. Fit the formulas (about a minute)
Needs r4's Test-B report (made by notebook 04, cell 3) for the size of the photo-model blur.
**Paste back:** everything this cell prints (three tables).


In [ ]:
!anthro ansur-fit --ansur {ANSUR} --report {RUN}/eval_report.json --out {FORMULAS}


## 5. Mix test for chest/bust and waist (a few minutes)
Decided on validation people only; a kept mix is then scored once on Test-B.
**Paste back:** everything this cell prints.


In [ ]:
!anthro ansur-mix --data {DATA_DIR} --ckpt {RUN}/best.pt --formulas {FORMULAS} --cache {CACHE} --geo-cache {GEO}


## 6. One Test-B woman, now with ANSUR rows

In [ ]:
from anthro.data import load_split
tb = load_split(DATA_DIR, "testB")
woman = tb[tb.gender == "female"].iloc[0]
man = tb[tb.gender == "male"].iloc[0]
for p in (woman, man):
    print(p.photo_id, p.gender, round(p.height_cm, 1), "cm", round(p.weight_kg, 1), "kg")


def mask(p, side):
    return f"{DATA_DIR}/testB/{'mask_left' if side else 'mask'}/{p.photo_id}.png"


In [ ]:
WF, WS = mask(woman, 0), mask(woman, 1)
WH, WW = f"{woman.height_cm:.1f}", f"{woman.weight_kg:.1f}"
!anthro predict-silhouettes --front {WF} --side {WS} --height {WH} --weight {WW} --sex female --age-range 25-34 --ckpt {RUN}/best.pt --ansur {FORMULAS}


## 7. Send the formula file
**Download `ansur_formulas.json`** (it is in `MyDrive/anthro/ansur2/`) and attach it in the chat. It holds
coefficients and summary errors only, no ANSUR person; it gets committed into the package (Ryan, 25 Nov).


## Before committing this notebook
**Edit → Clear all outputs**, then save. CI refuses notebooks with saved outputs (POLICY 6).
